In [1]:
#Task 3
# Modeli büyüt ve dropout ekle. Elindeki donanıma sığan bir konfigürasyon
# seç: n_embd, n_head, n_layer, block_size. Parametre sayısını, val loss'u
# ve eğitim süresini yaz, neden bu değerleri seçtiğini açıkla. Modelin
# ürettiği metinden bir örnek koy. (1:37:49 - 1:42:39)

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
device = 'cuda'
dropout = 0.2
n_head = 6
n_layer = 6
import time
import torch
import torch.nn as nn
from torch.nn import functional as F
with open("input.txt", "r", encoding = "utf-8") as f:
    text = f.read()

In [4]:
chars = sorted(list(set(text)))
vocab_size = len(chars)

In [5]:
#mapping
stoi = {ch:i for i,ch in enumerate(chars)}
itos = {i:ch for i,ch in enumerate(chars)}

In [6]:
#decoder & encoder
def encode(s):
    integers = [stoi[i] for i in s]
    return integers

def decode(l):
    lst = "".join([itos[i] for i in l])
    return lst

In [7]:
data = torch.tensor(encode(text), dtype=torch.long)

In [8]:
#train val split with 90% 10%
data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9*len(data))
train_data = data[:n]
val_data = data[n:]

In [9]:
#the way pick our batches
torch.manual_seed(1337)
batch_size = 4
block_size = 256

def get_batch(split):

    data = train_data if split == "train" else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,)) #do not want to grab latest datas like last data point as a starter
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x,y

In [10]:
#self attention head as a class
class Head(nn.Module):

    def __init__(self, head_size):
        super().__init__()
        self.head_size = head_size
        self.key = nn.Linear(emb_dim, head_size, bias = False)
        self.query = nn.Linear(emb_dim, head_size, bias = False)
        self.value = nn.Linear(emb_dim, head_size, bias = False)
        self.register_buffer("tril", torch.tril(torch.ones(block_size,block_size)))
        self.dropout = nn.Dropout(dropout)

    def forward(self,x):
        B,T,C =x.shape
        k = self.key(x)
        q = self.query(x)
        v = self.value(x)

        wei = q @ k.transpose(-2,-1) * self.head_size**-0.5
        wei = wei.masked_fill(self.tril[:T,:T] == 0, float("-inf"))
        wei = F.softmax(wei, dim=-1)
        wei = self.dropout(wei)

        out = wei @ v
        return out

In [11]:
#multi head self attention
class MultiHeadAttention(nn.Module):

    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(emb_dim, emb_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim = -1)
        out = self.dropout(self.proj(out))
        return out


In [12]:
#simple mlp with ReLU
class FeedForward(nn.Module):

    def __init__(self, emb_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(emb_dim,4 * emb_dim),
            nn.ReLU(),
            nn.Linear(4 * emb_dim, emb_dim),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)

class Block(nn.Module):

    def __init__(self,emb_dim, n_head):
        super().__init__()
        head_size = emb_dim // n_head
        self.slfattention_head = MultiHeadAttention(n_head, head_size)
        self.fddfrwd = FeedForward(emb_dim)
        self.ln1 = nn.LayerNorm(emb_dim)
        self.ln2 = nn.LayerNorm(emb_dim)

    def forward(self,x):
        x= x + self.slfattention_head(self.ln1(x))
        x= x + self.fddfrwd(self.ln2(x))
        return x

In [13]:
#model as a class
emb_dim = 384
class BigramLanguageModel(nn.Module):

    def __init__(self,vocab_size):
        super().__init__()

        self.token_embedding_table = nn.Embedding(vocab_size,emb_dim)   #token embeddings
        self.positional_embedding_table = nn.Embedding(block_size,emb_dim) #position embeddings
       ##adding our blocks
        self.blocks = nn.Sequential(*[Block(emb_dim, n_head=n_head) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(emb_dim)
        self.lm_head = nn.Linear(emb_dim, vocab_size)

    def forward(self, idx, targets=None):
        B,T = idx.shape
        tok_emb = self.token_embedding_table(idx)
        pos_emb = self.positional_embedding_table(torch.arange(T,device=device))
        x = tok_emb + pos_emb
        x = self.blocks(x)
        x = self.ln_f(x)
        logits = self.lm_head(x)

        if targets is None:
            loss =None
        else:
            B,T,C = logits.shape
            logits = logits.view(B*T,C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self,idx,max_new_tokens):

        for _ in range(max_new_tokens):

            idx_cond = idx[:,-block_size:]

            logits, loss =self(idx_cond)

            logits = logits[:,-1,:]

            probs = F.softmax(logits,dim=-1)

            idx_next = torch.multinomial(probs,num_samples=1)

            idx = torch.cat((idx,idx_next), dim=1)
        return idx

m = BigramLanguageModel(vocab_size).to(device)
sum(p.numel() for p in m.parameters())

10788929

In [14]:
#lets train our model
batch_size = 64
epoch = 5000

startt=time.time()
optimizer = torch.optim.AdamW(m.parameters(), lr=3e-4)

for _ in range(epoch):

    xb,yb = get_batch("train") #first gather batch
    logits, loss = m(xb,yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

print(f'the train loss is {loss.item()}')
end=time.time()
print(f'Training took {(end-startt)/60} minute')

the train loss is 1.1645373106002808
Training took 9.358502113819123 minute


In [15]:
#val loss estimator
m.eval()
@torch.no_grad()
def val_loss_estimator():
    losses = torch.zeros(200)
    for _ in range (200):
        x,y = get_batch("val")
        logits, loss = m(x,y)
        losses[_]= loss.item()
    print(f'the validation loss is around {losses.mean()}')
val_loss_estimator()

the validation loss is around 1.5035051107406616


In [16]:
#our tiny shakespeare which starts with ROMEO: haha
start = torch.tensor([encode("Romeo:")], dtype=torch.long, device=device)
out = m.generate(start, 2000)
print(decode(out[0].tolist()))

Romeo: I'll begun, sir, and a man thou wilt not;
if thou wert eyes to chaste, but it is there,
some two of two royal esteemer, in the old
estates, two traitories state, for made
three thou ask a grief man's gentleman:
true eye no part; and thy soldiers wembling when after,
steel in their gests, falsehood hearts; he hath presented
ear his brother France, and said 'twas young;'
And he would be thought upon thy right about youing?

RIVERS:
Command me to hear you.

CLIFFORD:Scotland, I would turn a cousin.

DUKE OF YORK:
Oy, my lord, and therefore be long for this.

QUEEN MARGARET:
Base bloody full of mine authority's pardon?
O else is no muddy uncle, he, hence, he twice bait
Her virtues, sir; here's no more dare so banish'd,
Which begin himself to you have marriage
But she would you do appear us.

QUEEN ELIZABETH:
What superfluity? what you dares my blood.

KING RICHARD III:
You told me have aught-shakes money than mine.
Then, beg's enemies, mildeth my souls;
Wherefore you shall stand but